# Lab: Web Scraping - Books to Scrape

**Objetivo:** Extraer datos de libros desde [Books to Scrape](https://books.toscrape.com/) usando `requests` y `BeautifulSoup`, y almacenar los resultados en un DataFrame de pandas filtrado por calificación mínima y precio máximo.

## Paso 1 - Importar librerías

Importamos las librerías necesarias para hacer peticiones web, parsear HTML y trabajar con datos.

In [2]:
# Instalamos las librerías necesarias si no están disponibles en el entorno
!pip install requests beautifulsoup4

     ---------------------------------------- 0.0/109.9 kB ? eta -:--:--
     -------------------------------------- 109.9/109.9 kB 3.2 MB/s eta 0:00:00



[notice] A new release of pip is available: 23.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Librería para hacer peticiones HTTP
import requests

# Librería para (leer y navegar) el HTML de la página
from bs4 import BeautifulSoup

# Librería para organizar los datos en un DataFrame
import pandas as pd

## Paso 2 - Explorar la página (una sola página)

Antes de construir el scraper completo, exploramos una sola página para entender la estructura HTML y encontrar los elementos que nos interesan.

In [4]:
# URL base del sitio
url = "https://books.toscrape.com/"

# Hacemos la petición HTTP para obtener el contenido de la página
response = requests.get(url)

# Comprobamos que la respuesta es correcta (200 = OK)
print(f"Código de respuesta: {response.status_code}")

Código de respuesta: 200


In [ ]:
# HTML con BeautifulSoup para poder navegar por su estructura
soup = BeautifulSoup(response.content, "html.parser")

# Vemos los primeros elementos para entender la estructura
print(soup.prettify()[:2000])

<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!-->
<html class="no-js" lang="en-us">
 <!--<![endif]-->
 <head>
  <title>
   All products | Books to Scrape - Sandbox
  </title>
  <meta content="text/html; charset=utf-8" http-equiv="content-type"/>
  <meta content="24th Jun 2016 09:29" name="created"/>
  <meta content="" name="description"/>
  <meta content="width=device-width" name="viewport"/>
  <meta content="NOARCHIVE,NOCACHE" name="robots"/>
  <!-- Le HTML5 shim, for IE6-8 support of HTML elements -->
  <!--[if lt IE 9]>
        <script src="//html5shim.googlecode.com/svn/trunk/html5.js"></script>
        <![endif]-->
  <link href="static/oscar/favicon.ico" rel="shortcut icon"/>
  <link href="static/oscar/css/styles.css" rel="stylesheet" type="tex

## Paso 3 - Explorar un libro individual


In [6]:
# Encontramos todos los libros de la página (cada uno está en un <article class='product_pod'>)
books = soup.find_all('article', class_='product_pod')

# Exploramos el primer libro para entender la estructura
first_book = books[0]
print(first_book.prettify())

<article class="product_pod">
 <div class="image_container">
  <a href="catalogue/a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_price">
  <p class="price_color">
   £51.77
  </p>
  <p class="instock availability">
   <i class="icon-ok">
   </i>
   In stock
  </p>
  <form>
   <button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">
    Add to basket
   </button>
  </form>
 </div>
</article>



In [7]:
# TÍTULO: está en el atributo 'title' del tag <a> dentro de <h3>
title = first_book.find('h3').find('a')['title']
print(f"Título: {title}")

Título: A Light in the Attic


In [8]:
# PRECIO: está en el tag <p class='price_color'>
# Eliminamos el símbolo '£' y convertimos a número
price_text = first_book.find('p', class_='price_color').text
price = float(price_text.replace('£', '').replace('Â', '').strip())
print(f"Precio: {price}")

Precio: 51.77


In [9]:
# CALIFICACIÓN: está en el tag <p> con clase que indica la valoración en palabras
# (One, Two, Three, Four, Five). La convertimos a número.

# Diccionario para convertir texto a número
rating_map = {
    'One': 1,
    'Two': 2,
    'Three': 3,
    'Four': 4,
    'Five': 5
}

# La clase del <p> de rating tiene dos partes: 'star-rating Three' → tomamos la segunda
rating_word = first_book.find('p', class_='star-rating')['class'][1]
rating = rating_map[rating_word]
print(f"Calificación: {rating}")

Calificación: 3


## Paso 4 - Función para extraer datos de una página

Encapsulamos la lógica de extracción en una función que recibe el objeto `soup` de una página y devuelve una lista de diccionarios con los datos de cada libro.

In [10]:
def scrape_books_from_page(soup):
    """
    Extrae los datos de todos los libros de una página ya revisada.
    Devuelve una lista de diccionarios con título, precio y calificación.
    """
    
    # Diccionario para convertir la calificación de texto a número
    rating_map = {
        'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5
    }
    
    # Lista donde guardaremos los datos de cada libro
    books_data = []
    
    # Encontramos todos los artículos (libros) de la página
    books = soup.find_all('article', class_='product_pod')
    
    # Iteramos por cada libro para extraer su información
    for book in books:
        
        # Extraemos el título desde el atributo 'title' del link
        title = book.find('h3').find('a')['title']
        
        # Extraemos el precio y lo limpiamos para convertirlo a float
        price_text = book.find('p', class_='price_color').text
        price = float(price_text.replace('£', '').replace('Â', '').strip())
        
        # Extraemos la calificación y la convertimos de texto a número
        rating_word = book.find('p', class_='star-rating')['class'][1]
        rating = rating_map[rating_word]
        
        # Guardamos los datos del libro como diccionario
        books_data.append({
            'title': title,
            'price': price,
            'rating': rating
        })
    
    return books_data

In [11]:
# Probamos la función con la primera página
test_data = scrape_books_from_page(soup)

# Vemos los primeros 3 resultados
test_data[:3]

[{'title': 'A Light in the Attic', 'price': 51.77, 'rating': 3},
 {'title': 'Tipping the Velvet', 'price': 53.74, 'rating': 1},
 {'title': 'Soumission', 'price': 50.1, 'rating': 1}]

## Paso 5 - Scraping de múltiples páginas

El sitio tiene 50 páginas. La URL cambia de forma incremental:
- Página 1: `https://books.toscrape.com/catalogue/page-1.html`
- Página 2: `https://books.toscrape.com/catalogue/page-2.html`
- ...

Usaremos un bucle para recorrer todas las páginas y acumular los datos.

In [12]:
# Lista donde acumulamos los datos de todas las páginas
all_books = []

# El sitio tiene 50 páginas en total
total_pages = 50

# Iteramos por cada página
for page_num in range(1, total_pages + 1):
    
    # Construimos la URL de cada página
    url = f"https://books.toscrape.com/catalogue/page-{page_num}.html"
    
    # Hacemos la petición HTTP
    response = requests.get(url)
    
    # Verificamos que la respuesta es correcta antes de seguir
    if response.status_code != 200:
        print(f"Error en página {page_num}: código {response.status_code}")
        break
    
    # Parseamos el HTML de la página
    soup = BeautifulSoup(response.content, "html.parser")
    
    # Extraemos los libros de esta página y los añadimos a la lista total
    page_books = scrape_books_from_page(soup)
    all_books.extend(page_books)
    
    # Indicamos el progreso cada 10 páginas
    if page_num % 10 == 0:
        print(f"Páginas procesadas: {page_num}/{total_pages} - Libros acumulados: {len(all_books)}")

print(f"\nScraping completado. Total de libros extraídos: {len(all_books)}")

Páginas procesadas: 10/50 - Libros acumulados: 200
Páginas procesadas: 20/50 - Libros acumulados: 400
Páginas procesadas: 30/50 - Libros acumulados: 600
Páginas procesadas: 40/50 - Libros acumulados: 800
Páginas procesadas: 50/50 - Libros acumulados: 1000

Scraping completado. Total de libros extraídos: 1000


## Paso 6 - Crear el DataFrame

Con todos los datos extraídos, creamos un DataFrame de pandas y lo exploramos brevemente.

In [13]:
# Creamos el DataFrame a partir de la lista de diccionarios
df_books = pd.DataFrame(all_books)

# Vemos las primeras filas para confirmar la estructura
df_books.head()

,title,price,rating
0,A Light in the Attic,51.77,3
1,Tipping the Velvet,53.74,1
2,Soumission,50.10,1
3,Sharp Objects,47.82,4
4,Sapiens: A Brief History of Humankind,54.23,5


In [14]:
# Revisamos la forma del DataFrame: filas y columnas
print(f"Dimensiones del DataFrame: {df_books.shape}")

# Revisamos los tipos de datos de cada columna
print("\nTipos de datos:")
print(df_books.dtypes)

# Revisamos que no hay valores nulos
print("\nValores nulos por columna:")
print(df_books.isnull().sum())

Dimensiones del DataFrame: (1000, 3)

Tipos de datos:
title      object
price     float64
rating      int64
dtype: object

Valores nulos por columna:
title     0
price     0
rating    0
dtype: int64


## Paso 7 - Filtrar por calificación mínima y precio máximo


In [16]:
# Definimos los criterios de filtrado
min_rating = 3   # Calificación mínima (en estrellas, de 1 a 5)
max_price = 20.0  # Precio máximo (en libras £)

# Aplicamos el filtro combinado con condición AND
df_filtered = df_books[
    (df_books['rating'] >= min_rating) &
    (df_books['price'] <= max_price)
]

# Reseteamos el índice para que sea continuo en el DataFrame filtrado
df_filtered = df_filtered.reset_index(drop=True)

print(f"Libros que cumplen los criterios (rating >= {min_rating} y precio <= {max_price}£): {len(df_filtered)}")
df_filtered.head(10)

Libros que cumplen los criterios (rating >= 3 y precio <= 20.0£): 116


,title,price,rating
0,The Coming Woman: A Novel Based on the Life of...,17.93,3
1,Set Me Free,17.46,5
2,The Four Agreements: A Practical Guide to Pers...,17.66,5
3,Sophie's World,15.94,5
4,Untitled Collection: Sabbath Poems 2014,14.27,4
5,Unicorn Tracks,18.78,3
6,This One Summer,19.49,4
7,Thirst,17.27,5
8,The Life-Changing Magic of Tidying Up: The Jap...,16.77,3
9,"Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Pr...",13.61,5


## Paso 8 - Exploración de los resultados

Revisamos  el DataFrame filtrado para confirmar que los datos son correctos.

In [17]:
# Estadísticas básicas del DataFrame filtrado
df_filtered.describe()

,price,rating
count,116.000000,116.000000
mean,14.466121,4.008621
std,2.714167,0.849508
min,10.000000,3.000000
25%,12.327500,3.000000
50%,14.145000,4.000000
75%,16.787500,5.000000
max,19.690000,5.000000


In [18]:
# Distribución de calificaciones en el DataFrame filtrado
print("Distribución de calificaciones:")
print(df_filtered['rating'].value_counts().sort_index())

Distribución de calificaciones:
rating
3    41
4    33
5    42
Name: count, dtype: int64


In [19]:
# DataFrame final con todos los libros que cumplen los criterios
print(f"DataFrame final: {df_filtered.shape[0]} libros con rating >= {min_rating} estrellas y precio <= {max_price}£")
df_filtered

DataFrame final: 116 libros con rating >= 3 estrellas y precio <= 20.0£


,title,price,rating
0,The Coming Woman: A Novel Based on the Life of...,17.93,3
1,Set Me Free,17.46,5
2,The Four Agreements: A Practical Guide to Pers...,17.66,5
3,Sophie's World,15.94,5
4,Untitled Collection: Sabbath Poems 2014,14.27,4
...,...,...,...
111,The Edge of Reason (Bridget Jones #2),19.18,4
112,The Complete Maus (Maus #1-2),10.64,3
113,The Communist Manifesto,14.76,3
114,Sister Sable (The Mad Queen #1),13.33,3
